## SQL Server PlugIN

# Confirm you have the driver installed

In [52]:
import pyodbc
import os
from dotenv import load_dotenv  
import pandas as pd
from sqlalchemy import create_engine

load_dotenv(override=True) 

True

In [53]:
import os
import pyodbc
from sqlalchemy import create_engine

# Get environment variables
username = os.getenv("SQL_USERNAME")
password = os.getenv("SQL_PASSWORD")    
server = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")

# Check for available drivers
available_drivers = pyodbc.drivers()
preferred_drivers = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"]

# Select the first available preferred driver
selected_driver = next((driver for driver in preferred_drivers if driver in available_drivers), None)

if not selected_driver:
    raise Exception("Neither ODBC Driver 17 nor 18 for SQL Server is installed.")

# Create the SQLAlchemy engine
print("*******************")
engine = create_engine(
    f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver={selected_driver.replace(' ', '+')}"
)



query = "SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'" # Replace with your actual query
df = pd.read_sql(query, engine)
html_output = df.to_html(index=False, border=0, justify='center', classes='table table-striped table-bordered')
print(html_output)

*******************
<table class="dataframe table table-striped table-bordered">
  <thead>
    <tr style="text-align: center;">
      <th>TABLE_SCHEMA</th>
      <th>TABLE_NAME</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>dbo</td>
      <td>Capabilities</td>
    </tr>
    <tr>
      <td>dbo</td>
      <td>Incidents</td>
    </tr>
    <tr>
      <td>dbo</td>
      <td>Units</td>
    </tr>
    <tr>
      <td>dbo</td>
      <td>UnitCapabilities</td>
    </tr>
    <tr>
      <td>dbo</td>
      <td>IncidentNotes</td>
    </tr>
  </tbody>
</table>


In [54]:
# Note: if using a virtual environment, do not run this cell
#%pip install -U semantic-kernel
from semantic_kernel import __version__

__version__

'1.33.0'

In [55]:
#from services import Service
from service_settings import ServiceSettings
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import KernelArguments


from semantic_kernel import Kernel
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.functions.kernel_arguments import KernelArguments
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)

from semantic_kernel.contents import ChatMessageContent, TextContent, ImageContent
from semantic_kernel.contents.utils.author_role import AuthorRole

import os
from Plugin.SQLPlugin import SQLPlugin


In [56]:

kernel = Kernel()
service_id = "default"
kernel.add_service(AzureChatCompletion(service_id=service_id,),)
kernel.add_plugin(SQLPlugin(),plugin_name="SQL",)


KernelPlugin(name='SQL', description=None, functions={'check_installed_drivers': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='check_installed_drivers', plugin_name='SQL', description='check installed drivers if hanving issue connecting to SQL SErver', parameters=[KernelParameterMetadata(name='driver', description='The input driver', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': 'The input driver'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='The output is a string', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': 'The output is a string'}, include_in_function_choices=True), additional_properties={}), invocation_duration_histogram=<opentelemetry.metrics._internal.instrument._ProxyHistogram object at 0x000001A85CD0E2E0>

In [57]:
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior


chat_completion : AzureChatCompletion = kernel.get_service(type=ChatCompletionClientBase)

# Enable planning
execution_settings = AzureChatPromptExecutionSettings(tool_choice="auto")
execution_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()
# Create a history of the conversation
history = ChatHistory()
history.add_system_message("You are a helpful assistant who can write sql to answer questions.  If you can't connect, troubleshoot the connection.")
history.add_user_message("Can tell me about the tables in the database?")

result = (await chat_completion.get_chat_message_contents(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
        arguments=KernelArguments(),
    ))[0]
print(result)


history.add_assistant_message(str(result))


Here’s an overview of the objects in the dbo schema:

1. Tables

  • Capabilities  
    – CapabilityCode (nvarchar)  
    – Description (nvarchar)

  • Units  
    – UnitKey (uniqueidentifier)  
    – UnitId (nvarchar)  
    – AssignedArea (nvarchar)  
    – Available (bit)  
    – Status (nvarchar)  
    – LastKnownLatitude (float)  
    – LastKnownLongitude (float)  
    – AssignedIncident (uniqueidentifier)

  • Incidents  
    – IncidentKey (uniqueidentifier)  
    – IncidentId (nvarchar)  
    – Latitude (float)  
    – Longitude (float)  
    – Timestamp (datetime)  
    – Status (nvarchar)

  • UnitCapabilities  
    – UnitKey (uniqueidentifier)  
    – CapabilityCode (nvarchar)

  • IncidentNotes  
    – NoteId (uniqueidentifier)  
    – IncidentKey (uniqueidentifier)  
    – Timestamp (datetime)  
    – NoteText (nvarchar)

2. View

  • vw_UnitAvailability  
    – UnitId (nvarchar)  
    – AssignedArea (nvarchar)  
    – Available (bit)  
    – Status (nvarchar)  
    – LastKn

In [58]:

# Create a history of the conversation
history.add_user_message("what is the schema of the Incidents table?")

result = (await chat_completion.get_chat_message_contents(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
        arguments=KernelArguments(),
    ))[0]
print(result)


The Incidents table (dbo.Incidents) has the following columns:

• IncidentKey  
  – Data type: uniqueidentifier  
  – Nullable: NO  

• IncidentId  
  – Data type: nvarchar(50)  
  – Nullable: YES  

• Latitude  
  – Data type: float  
  – Nullable: YES  

• Longitude  
  – Data type: float  
  – Nullable: YES  

• Timestamp  
  – Data type: datetime  
  – Nullable: YES  

• Status  
  – Data type: nvarchar(10)  
  – Nullable: YES


In [60]:
history.add_user_message("For the unassigned incidents that are open, assign based on location, and on the capabilities of the agents looking at the notes for the incidnets.  Review the schema of tables before making sql statements to ensure quality SQL select staements.  Provide suggested assignements, and reasons why")

result = (await chat_completion.get_chat_message_contents(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
        arguments=KernelArguments(),
    ))[0]
print(result)

I’ve identified all open, unassigned incidents, extracted any required capabilities from their notes, and matched them to available units:

• INC-00082  
  – Suggested unit: UNIT-016 (nearest available)  
  – Distance ≈ 58.18  
  – Reason: No specific capabilities required; nearest available unit

• INC-00083  
  – UNIT-011 at ~70.46 units away  
  – No specific capabilities required

• INC-00090  
  – UNIT-016 at ~81.10 units away  
  – No specific capabilities required

• INC-00091  
  – UNIT-020 at ~48.85 units away  
  – No specific capabilities required

• INC-00092  
  – UNIT-017 at ~38.01 units away  
  – No specific capabilities required

• INC-00093  
  – UNIT-014 at ~11.24 units away  
  – No specific capabilities required

• INC-00094  
  – UNIT-017 at ~53.96 units away  
  – No specific capabilities required

• INC-00096  
  – UNIT-020 at ~125.03 units away  
  – No specific capabilities required

• INC-00097  
  – UNIT-014 at ~32.65 units away  
  – No specific capabilitie